# <font color = 'red'> DEPENDENCIAS

In [23]:
import pandas as pd
import numpy as np
import plotly.express as px
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.preprocessing import MinMaxScaler

import sys
import os

# Agregar la carpeta calibration_code al path
sys.path.append(os.path.abspath("../../calibration_code"))

# Ahora puedes importar los módulos personalizados
from modelling_tools import (plot_histogram, plot_univariate_freq, assign_deciles, count_categories_by_decile, 
                             calculate_category_proportions, summarize_decile_analysis, summarize_grouped_deciles, group_deciles,
                             compute_odds_ratio)
from visualization_tools import plot_interactive_chart
from utils import g
from config import get_data_path, get_code_path
from data_cleaning import check_dataframe_quality

# <font color = 'red'> CARGA DE DATOS

In [24]:
df = pd.read_csv(get_data_path("bivariate_preprocessed_data.csv"))

In [25]:
res = check_dataframe_quality(df)

No missing values found.
No infinite values found.
No duplicate rows found.


# <font color = 'red'> ANÁLISIS

In [26]:
col = "Annual_Income"

## <font color = 'skyblue'> ANÁLISIS GENERAL

La mediana de los ingresos de los clientes malos parecen significativamente inferiores a los clientes Standard y en mayor medida a los Buenos.

In [27]:
fig_box = px.box(df, x="Credit_Mix", y=col, title=f"Distribution of {col} by Credit Score Category")
fig_box.show()

## <font color = 'skyblue'> ANÁLISIS POR DECILES

In [28]:
continuous_variable= col
decile_col_name = continuous_variable + '_Decile'
target_col_string = "Credit_Mix" # variable dependiente con nombres string
target_col = 'Credit_Score' # variable dependiente int (para modelos)

In [29]:
analysis_summary = summarize_decile_analysis(df, continuous_variable, decile_col_name, target_col_string)

# Obtener los resultados
df_deciles = analysis_summary["df_deciles"]  # DataFrame con los deciles asignados
deciles_summary = analysis_summary["decile_summary"]  # Resumen de deciles con conteos y proporciones
display(deciles_summary)
res = check_dataframe_quality(df_deciles)

,Decile_Min,Decile_Max,Decile_Count,Decile_Proportion,count_Bad,count_Good,count_Standard,prop_Bad,prop_Good,prop_Standard
Annual_Income_Decile,,,,,,,,,,
0,7005.930,14263.355,10000,0.1,4384,576,5040,0.4384,0.0576,0.5040
1,14264.460,17592.340,10000,0.1,3880,1872,4248,0.3880,0.1872,0.4248
2,17594.015,21075.820,10000,0.1,3944,1976,4080,0.3944,0.1976,0.4080
3,21085.670,30988.540,10000,0.1,808,3024,6168,0.0808,0.3024,0.6168
4,30988.610,36996.830,10000,0.1,1800,3344,4856,0.1800,0.3344,0.4856
5,37002.580,48434.660,10000,0.1,2472,4272,3256,0.2472,0.4272,0.3256
6,48494.550,63561.980,10000,0.1,3656,1528,4816,0.3656,0.1528,0.4816
7,63563.880,81049.770,10000,0.1,2288,2632,5080,0.2288,0.2632,0.5080
8,81066.720,108782.520,10000,0.1,536,4160,5304,0.0536,0.4160,0.5304


No missing values found.
No infinite values found.
No duplicate rows found.


In [30]:
df[(df[continuous_variable] >= 17594.015) & (df[continuous_variable] <= 21075.820) & (df['Credit_Score'] == 2)].shape

(1976, 85)

Porporción de Buenos: aunque se observa una relación positiva entre los ingresos netos y la proporción de buenos, se observa un
cambio abrupto entre los deciles 6 y 7, al pasar de una proporción de buenos de 41% a 15%,
lo cual no es razonable.

Proporción de standard: aunque sí existe una tendencia negativa entre la proporción de Standard y los ingresos
hay cambios irregulares que son poco razonables.

Proporción de malos: de manera similar, la proporción de malos y los ingresos tienen una relación inversa, sin embargo, 
hay cambios irregulares abruptos.

In [31]:
chart_types = {
    "prop_Good":"line",
    "prop_Standard":"line",
    "prop_Bad": "line",  
    "Decile_Count": "bar"    
}

fig = plot_interactive_chart(
    df=deciles_summary,  
    y_columns=["prop_Bad", "prop_Good", "prop_Standard", "Decile_Count"],  
    x_column="Decile_Max",  
    chart_types=chart_types, 
    title=f"Proportion of Credit Score Categories by {continuous_variable}",
    x_title="Decile",
    y_title="Decile Count",  
    y2_title="Proportion",   
    secondary_y=["prop_Bad", "prop_Good", "prop_Standard"],  
    width=900,
    height=500,
    custom_colors={"prop_Bad": "red", "prop_Good":"lightgreen",
    "prop_Standard":"brown", "Decile_Count": "gray"}  
)

fig.show()

<font color = 'brown'> Agrupación de deciles

Se agrupan deciles buscando una relación monótona entre las proporciones y los ingresos:

In [32]:
group_map = {0: "Group_1", 
             1: "Group_2", 
             2: "Group_2", 
             3: "Group_3", 
             4: "Group_3",
             5: "Group_3", 
             6: "Group_3", 
             7: "Group_3", 
             8: "Group_3", 
             9: "Group_3"}

# Agrupar los deciles
grouped_col_name = "Grouped_" + continuous_variable

df_deciles_grouped = group_deciles(df_deciles, decile_col_name, grouped_col_name, group_map)

summary_results = summarize_grouped_deciles(df_deciles_grouped, grouped_col_name, continuous_variable, target_col_string, prefix="Decile_")
grouped_deciles_summary = summary_results['df']


# Definir el mapeo manual de los grupos a enteros
group_mapping = {
    'Group_1': 1,
    'Group_2': 2,
    'Group_3': 3
}

if not set(group_map.values()) == set(group_mapping.keys()):
    print("Problemas en el mapeo de grupos a enteros!")

# Usar `.map()` en lugar de `.replace()` para evitar el warning
df_deciles_grouped[grouped_col_name] = (
    df_deciles_grouped[grouped_col_name]
    .map(group_mapping)  # Mapear los valores
    .astype("Int64")      # Convertir a entero manejando NaN si existen
)

display(grouped_deciles_summary)

res = check_dataframe_quality(df_deciles_grouped)

,Decile_Min,Decile_Max,Decile_Count,Decile_Proportion,count_Bad,count_Good,count_Standard,prop_Bad,prop_Good,prop_Standard
Grouped_Annual_Income,,,,,,,,,,
Group_1,7005.93,14263.355,10000,0.1,4384,576,5040,0.438400,0.057600,0.5040
Group_2,14264.46,21075.820,20000,0.2,7824,3848,8328,0.391200,0.192400,0.4164
Group_3,21085.67,179987.280,70000,0.7,11560,25960,32480,0.165143,0.370857,0.4640


No missing values found.
No infinite values found.
No duplicate rows found.


In [33]:
chart_types = {
    "prop_Good":"line",
    "prop_Standard":"line",
    "prop_Bad": "line",  
    "Decile_Count": "bar"    
}

fig = plot_interactive_chart(
    df=grouped_deciles_summary,  
    y_columns=["prop_Bad", "prop_Good", "prop_Standard", "Decile_Count"],  
    x_column="Decile_Max",  
    chart_types=chart_types, 
    title=f"Proportion of Credit Score Categories by {continuous_variable}",
    x_title="Decile",
    y_title="Decile Count",  
    y2_title="Proportion",   
    secondary_y=["prop_Bad", "prop_Good", "prop_Standard"],  
    width=900,
    height=500,
    custom_colors={"prop_Bad": "red", "prop_Good":"lightgreen",
    "prop_Standard":"brown", "Decile_Count": "gray"}  
)

fig.show()

## <font color = 'skyblue'> REGRESIONES BIVARIADAS

Como Credit_Score tiene tres categorías (Bad, Standard, Good) se pueden usar dos enfoques 
de regresión categórica: 

- Regresión Logística Multinomial → No asume orden en las categorías (como si fueran colores: rojo, azul, verde).
- Regresión Logística Ordinal → Asume que hay un orden en las categorías (Bad < Standard < Good).

Dado que hay un orden entre las categorías se utiliza Regresión Logística Ordinal:

<font color = 'gold'> Sin Agrupaciones

In [34]:
df_ = df_deciles_grouped.copy()
x_variable = continuous_variable
res = check_dataframe_quality(df_)

No missing values found.
No infinite values found.
No duplicate rows found.


In [35]:
# para no generar problemas numéricos debe escalarse esta variable.
# se opta por normalizar la variable:

g(df_[[x_variable]].describe()).transpose()

,count,mean,std,min,25%,50%,75%,max
Annual_Income,"100,000.00","50,505.12","38,299.42","7,005.93","19,342.97","36,999.71","71,683.47","179,987.28"


Todos los coeficientes son significativos.

Annual_Income_Decile 3.1251: según lo esperado, el coeficiente es positivo: por cada dólar adicional, la probabilidad de estar en una categoría superior de Credit_Score aumenta.

Threshold 0/1 -0.4888: Umbral que separa las categorías Bad y Standard. Si la puntuación supera este umbral, es más probable que 
sea Good en lugar de Standard.

Threshold 1/2 0.7819: Umbral que separa las categorías Standard y Good. Si la puntuación supera este umbral, es más probable que 
sea Good en lugar de Standard.

In [40]:
df_ = df_deciles_grouped.copy()
x_variable = continuous_variable

scaler = MinMaxScaler()
df_[continuous_variable + "_Scaled"] = scaler.fit_transform(df_[[x_variable]])

res = check_dataframe_quality(df_)

# Ajustar el modelo de regresión logística ordinal con la variable escalada
model_income = OrderedModel(df_[target_col], df_[continuous_variable + "_Scaled"], distr="logit")
result_income = model_income.fit(method='bfgs')

# Mostrar resumen del modelo
print(result_income.summary())

# Calcular e interpretar el Odds Ratio
res_odds = compute_odds_ratio(result_income, variable_name=continuous_variable + "_Scaled", description="Ingreso Anual Escalado")
print(res_odds["interpretation"])


No missing values found.
No infinite values found.
No duplicate rows found.
Optimization terminated successfully.
         Current function value: 0.998728
         Iterations: 11
         Function evaluations: 13
         Gradient evaluations: 13
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:                -99873.
Model:                   OrderedModel   AIC:                         1.998e+05
Method:            Maximum Likelihood   BIC:                         1.998e+05
Date:                Sat, 29 Mar 2025                                         
Time:                        12:42:11                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                           coef    std er

<font color = 'gold'> Por Deciles

Todos los coeficientes son significativos.

Annual_Income_Decile	0.2183: Por cada decil adicional, la probabilidad de estar en una categoría superior de Credit_Score aumenta.

Threshold 0/1	-0.2842: Umbral que separa las categorías Bad y Standard: un cliente con una puntuación de regresión 
superior a -0.2842: es más probable que sea Standard que Bad.

Threshold 1/2	0.7758: Umbral que separa las categorías Standard y Good. Si la puntuación supera 0.7758, es más probable que 
sea Good en lugar de Standard.

In [37]:
df_ = df_deciles_grouped.copy()
x_variable = decile_col_name

res = check_dataframe_quality(df_)

model_age_decile = OrderedModel(df_[target_col], df_[x_variable], distr="logit")

result_age_decile = model_age_decile.fit(method='bfgs')

print(result_age_decile.summary())

res_odds = compute_odds_ratio(result_age_decile, variable_name = x_variable, description = 'Decil de Ingreso Neto')
print(res_odds['interpretation'])

No missing values found.
No infinite values found.
No duplicate rows found.
Optimization terminated successfully.
         Current function value: 1.007593
         Iterations: 10
         Function evaluations: 12
         Gradient evaluations: 12
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:            -1.0076e+05
Model:                   OrderedModel   AIC:                         2.015e+05
Method:            Maximum Likelihood   BIC:                         2.016e+05
Date:                Sat, 29 Mar 2025                                         
Time:                        12:34:59                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                           coef    std er

<font color = 'gold'> Por Agrupamientos de Deciles

Todos los coeficientes son significativos.

Grouped_Monthly_Inhand_Salary 0.5537: Por cada grupo adcional, la probabilidad de estar en una categoría superior de Credit_Score aumenta.

Threshold 0/1 0.7172: Umbral que separa las categorías Bad y Standard.

Threshold 1/2	0.7862: Umbral que separa las categorías Standard y Good. Si la puntuación supera 0.7604, es más probable que 
sea Good en lugar de Standard.

In [38]:
df_ = df_deciles_grouped.copy()
x_variable = grouped_col_name

df_[target_col] = df_[target_col].astype(int)
df_[x_variable] = df_[x_variable].astype(int)

res = check_dataframe_quality(df_)

model_age_decile = OrderedModel(df_[target_col], df_[x_variable], distr="logit")

result_age_decile = model_age_decile.fit(method='bfgs')

print(result_age_decile.summary())

res_odds = compute_odds_ratio(result_age_decile, variable_name = x_variable, description = 'Decil de Edad')
print(res_odds['interpretation'])

No missing values found.
No infinite values found.
No duplicate rows found.
Optimization terminated successfully.
         Current function value: 1.017918
         Iterations: 13
         Function evaluations: 15
         Gradient evaluations: 15
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:            -1.0179e+05
Model:                   OrderedModel   AIC:                         2.036e+05
Method:            Maximum Likelihood   BIC:                         2.036e+05
Date:                Sat, 29 Mar 2025                                         
Time:                        12:35:01                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                            coef    std e

## <font color = 'skyblue'> CONCLUSIONES

### 📊 Comparación de Representaciones de `Annual_Income` usando Regresión Ordinal

| Representación                     | Coeficiente principal | Indicadores de ajuste                                                               | Interpretación                                                                                                                                   |
|-----------------------------------|------------------------|--------------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------------------------------------------------------------|
| `Annual_Income_Scaled`            | 3.1251                 | **Log-Likelihood**: -99,873<br>**AIC**: 199,746<br>**BIC**: 199,788                 | 🔹 Modelo con mejor ajuste global.<br>🔹 Muy informativa, conserva el detalle continuo sin problemas de escala.                                 |
| `Annual_Income_Decile`            | 0.2183                 | **Log-Likelihood**: -100,759<br>**AIC**: 201,519<br>**BIC**: 201,561                | 🔹 Peor ajuste comparado con las otras opciones.<br>🔹 La discretización en deciles reduce la información disponible.                            |
| `Grouped_Annual_Income`           | 0.8407                 | **Log-Likelihood**: -101,792<br>**AIC**: 203,584<br>**BIC**: 203,625                | 🔹 Ajuste más pobre.<br>🔹 Aunque mejora la interpretabilidad, los **umbrales no son monótonos**, lo que puede afectar la validez del modelo.   |


## <font color = 'skyblue'> EXPORTACIÓN DE DATOS CON VARIABLES ADICIONALES

In [39]:
# df_deciles_grouped.to_csv("../../calibration_data/preprocessed_data.csv", index=False)
